# OSL Connectome2 All-in-One Notebook

This notebook is fully in one file style.
It embeds core code directly:
- `odor_env_v4.py`
- `networks.py`
- `rsac_agent.py`
- `buffer.py`
- train/eval loop

Run from top to bottom.


In [ ]:
# If needed, install dependencies once:
# !pip install -r requirements.txt


In [ ]:
import random

import numpy as np
import torch


def set_global_seed(seed, deterministic_torch=True):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic_torch:
        if hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except TypeError:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass


In [ ]:
import random
import numpy as np

class EpisodeReplayBuffer:
    def __init__(self, cap_steps=150000):
        self.cap_steps = int(cap_steps)
        self.episodes = [] 
        self.n_steps = 0

    def add_episode(self, ep):
        """ep: list of (obs, action, reward, next_obs, terminal)."""
        if not ep: return
        self.episodes.append(ep)
        self.n_steps += len(ep)
        while self.n_steps > self.cap_steps and self.episodes:
            old = self.episodes.pop(0)
            self.n_steps -= len(old)

    def __len__(self):
        return self.n_steps

    def sample(self, batch_size, seq_len):
        candidates = [ep for ep in self.episodes if len(ep) >= seq_len]
        if not candidates:
            raise RuntimeError("Buffer insufficient for sampling sequences.")

        obs_seqs, act_seqs, rew_seqs, terminal_seqs = [], [], [], []

        for _ in range(batch_size):
            ep = random.choice(candidates)
            s = random.randint(0, len(ep) - seq_len)
            chunk = ep[s:s + seq_len]

            o0 = chunk[0][0]
            obs_seq = [o0] + [tr[3] for tr in chunk] # Next obs including
            
            obs_seqs.append(np.asarray(obs_seq, dtype=np.float32))
            act_seqs.append(np.asarray([tr[1] for tr in chunk], dtype=np.int64))
            rew_seqs.append(np.asarray([tr[2] for tr in chunk], dtype=np.float32))
            terminal_seqs.append(np.asarray([tr[4] for tr in chunk], dtype=np.float32))

        return (
            np.stack(obs_seqs, axis=0),
            np.stack(act_seqs, axis=0),
            np.stack(rew_seqs, axis=0),
            np.stack(terminal_seqs, axis=0),
        )

    def sample_continuous(self, batch_size, seq_len):
        candidates = [ep for ep in self.episodes if len(ep) >= seq_len]
        if not candidates:
            raise RuntimeError("Buffer insufficient for sampling sequences.")

        obs_seqs, act_seqs, rew_seqs, terminal_seqs = [], [], [], []

        for _ in range(batch_size):
            ep = random.choice(candidates)
            s = random.randint(0, len(ep) - seq_len)
            chunk = ep[s:s + seq_len]

            o0 = chunk[0][0]
            obs_seq = [o0] + [tr[3] for tr in chunk]

            obs_seqs.append(np.asarray(obs_seq, dtype=np.float32))
            act_seqs.append(np.asarray([tr[1] for tr in chunk], dtype=np.float32))
            rew_seqs.append(np.asarray([tr[2] for tr in chunk], dtype=np.float32))
            terminal_seqs.append(np.asarray([tr[4] for tr in chunk], dtype=np.float32))

        return (
            np.stack(obs_seqs, axis=0),
            np.stack(act_seqs, axis=0),
            np.stack(rew_seqs, axis=0),
            np.stack(terminal_seqs, axis=0),
        )


In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces


class OdorHoldEnvV4(gym.Env):
    metadata = {"render_modes": ["rgb_array"], "render_fps": 30}

    def __init__(
        self,
        render_mode=None,
        L=3.0,
        dt=0.1,
        src_x=0.0,
        src_y=0.0,
        wind_x=1.0,
        wind_y=0.0,
        sensor_offset=0.08,
        sigma_c=1.0,
        sigma_r=0.8,
        r_goal=0.35,
        b_hold=0.5,
        b_oob=5.0,
        max_steps=300,
        seed=0,
        bg_c=0.0,
        sensor_noise=0.01,
        v_min=0.0,
        v_max=0.45,
        omega_max=5.0,
        accel_max=1.2,
        omega_accel_max=50.0,
        control_penalty=0.01,
        turn_penalty=0.01,
        cast_penalty=0.025,
        reward_mode="mechanical",
        bio_reward_scale=0.5,
        goal_hold_steps=20,
        terminate_on_hold=True,
        turn_requires_cast=True,
        turn_window_steps=1,
    ):
        super().__init__()
        self.render_mode = render_mode
        self.L = float(L)
        self.dt = float(dt)
        self.ds = float(sensor_offset)
        self.src_x = float(np.clip(float(src_x), -self.L, self.L))
        self.src_y = float(np.clip(float(src_y), -self.L, self.L))
        self.wind_x = float(wind_x)
        self.wind_y = float(wind_y)
        self._wind_mag = float(np.hypot(self.wind_x, self.wind_y))

        if self._wind_mag > 1e-6:
            self._wind_dir = (self.wind_x / self._wind_mag, self.wind_y / self._wind_mag)
        else:
            self._wind_dir = (0.0, 0.0)

        self.sigma_c = float(sigma_c)
        self.sigma_r = float(sigma_r)
        self.r_goal = float(r_goal)
        self.b_hold = float(b_hold)
        self.b_oob = float(b_oob)
        self.max_steps = int(max_steps)
        self.bg_c = float(bg_c)
        self.sensor_noise = float(sensor_noise)

        self.v_min = float(v_min)
        self.v_max = float(v_max)
        self.omega_max = float(omega_max)
        self.accel_max = float(accel_max)
        self.omega_accel_max = float(omega_accel_max)
        self.control_penalty = float(control_penalty)
        self.turn_penalty = float(turn_penalty)
        self.cast_penalty = float(cast_penalty)
        self.bio_reward_scale = float(max(0.0, bio_reward_scale))
        mode = str(reward_mode).lower()
        # Backward compatibility with old configs.
        if mode == "dense":
            mode = "mechanical"
        self.reward_mode = mode
        if self.reward_mode not in ("mechanical", "bio"):
            raise ValueError("reward_mode must be one of ['mechanical', 'bio']")
        self.goal_hold_steps = int(max(1, goal_hold_steps))
        self.terminate_on_hold = bool(terminate_on_hold)
        self.turn_requires_cast = bool(turn_requires_cast)
        self.turn_window_steps = int(max(1, turn_window_steps))

        # action = [v_cmd, omega_cmd, cast_cmd]
        self.action_space = spaces.Box(
            low=np.array([self.v_min, -self.omega_max, 0.0], dtype=np.float32),
            high=np.array([self.v_max, self.omega_max, 1.0], dtype=np.float32),
            dtype=np.float32,
        )

        # Per-step observation for both reward modes: [concentration, mode]
        # mode: 0=run, 1=cast
        self.obs_step_dim = 2
        obs_low = np.array([0.0, 0.0], dtype=np.float32)
        obs_high = np.array([1.0, 1.0], dtype=np.float32)
        self.observation_space = spaces.Box(low=obs_low, high=obs_high, dtype=np.float32)

        self.np_random = np.random.default_rng(seed)

        self._step = 0
        self.x = 0.0
        self.y = 0.0
        self.th = 0.0
        self.v = 0.0
        self.omega = 0.0

        self.in_cast = False
        self.cast_phase = 0
        self.turn_steps_left = 0
        self._scan_dirs = np.array([np.pi / 2, -np.pi / 2], dtype=np.float32)
        self._scan_seq = (0, 1, 0, 1)
        self._scan_c = np.zeros(4, dtype=np.float32)
        self.goal_hold_count = 0
        self.prev_in_goal = False

        self._sense_pt = None
        self._trail = []
        self._img_size = 360
        self._heatmap_img = None
        self._cbar_img = None
        self._last_obs = np.zeros((self.obs_step_dim,), dtype=np.float32)

    def _conc(self, x, y):
        x = np.asarray(x, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)
        xr = x - np.float32(self.src_x)
        yr = y - np.float32(self.src_y)

        if self._wind_mag <= 1e-6:
            r2 = xr * xr + yr * yr
            c = np.exp(-r2 / (2.0 * self.sigma_c * self.sigma_c))
        else:
            wx, wy = self._wind_dir
            t = xr * wx + yr * wy
            s = -xr * wy + yr * wx
            stretch = 1.0 + min(self._wind_mag, 2.0)
            sigma_s = self.sigma_c
            sigma_t = self.sigma_c * stretch
            sigma_up = self.sigma_c / stretch
            t_pos = np.maximum(0.0, t)
            t_neg = np.maximum(0.0, -t)
            c = np.exp(-((s * s) / (2.0 * sigma_s ** 2) + (t_pos ** 2) / (2.0 * sigma_t ** 2) + (t_neg ** 2) / (2.0 * sigma_up ** 2)))

        c = self.bg_c + (1.0 - self.bg_c) * c
        return np.clip(c, 0.0, 1.0)

    def _sense(self, phi=0.0):
        ang = float(self.th + phi)
        sx = float(self.x + np.cos(ang) * self.ds)
        sy = float(self.y + np.sin(ang) * self.ds)
        c = self._conc(sx, sy)
        if self.sensor_noise > 0:
            c += float(self.np_random.normal(0, self.sensor_noise))
        self._sense_pt = (sx, sy)
        return float(np.clip(c, 0.0, 1.0))

    def _set_obs(self, c, mode):
        self._last_obs = np.array([float(c), float(mode)], dtype=np.float32)

    def _get_obs(self):
        return self._last_obs.copy()

    def _norm_angle(self, a):
        return (a + np.pi) % (2 * np.pi) - np.pi

    def _cast_step(self):
        i = int(self.cast_phase)
        side = int(self._scan_seq[i])
        phi = float(self._scan_dirs[side])
        c = self._sense(phi)
        self._scan_c[i] = float(c)

        self.cast_phase += 1
        if self.cast_phase >= 4:
            self.in_cast = False
            self.cast_phase = 0
            self._scan_c[:] = 0.0
            self.turn_steps_left = self.turn_window_steps
        return c

    def reset(self, seed=None, options=None):
        del options
        if seed is not None:
            self.np_random = np.random.default_rng(seed)

        spawn_margin = 0.25
        spawn_radius_tries = 80
        spawn_angle_tries = 80
        r_min, r_max = max(self.r_goal + spawn_margin, 0.6), min(0.8 * self.L, self.L - spawn_margin)
        cx, cy = self.src_x, self.src_y

        x0, y0 = float(cx), float(cy)
        found = False
        for _ in range(max(1, int(spawn_radius_tries))):
            r0 = float(self.np_random.uniform(r_min, r_max))
            for _ in range(max(1, int(spawn_angle_tries))):
                ang = float(self.np_random.uniform(-np.pi, np.pi))
                tx = float(cx + r0 * np.cos(ang))
                ty = float(cy + r0 * np.sin(ang))
                if (-self.L + spawn_margin) <= tx <= (self.L - spawn_margin) and (-self.L + spawn_margin) <= ty <= (self.L - spawn_margin):
                    x0, y0 = tx, ty
                    found = True
                    break
            if found:
                break
        if not found:
            x0 = float(np.clip(cx, -self.L + spawn_margin, self.L - spawn_margin))
            y0 = float(np.clip(cy, -self.L + spawn_margin, self.L - spawn_margin))

        self.x, self.y = float(x0), float(y0)
        self.th = self.np_random.uniform(-np.pi, np.pi)
        self.v = 0.0
        self.omega = 0.0
        self._step = 0
        self.in_cast = False
        self.cast_phase = 0
        self.turn_steps_left = 0
        self._scan_c[:] = 0.0
        self.goal_hold_count = 0
        self.prev_in_goal = False
        self._trail = [(self.x, self.y)]

        c0 = self._sense(0.0)
        self._set_obs(c0, mode=0.0)
        return self._get_obs(), {}

    def step(self, action):
        self._step += 1
        prev_c = float(self._last_obs[0])
        prev_in_goal = bool(self.prev_in_goal)
        a = np.asarray(action, dtype=np.float32)
        a = np.clip(a, self.action_space.low, self.action_space.high)
        v_cmd = float(a[0])
        omega_cmd = float(a[1])
        cast_cmd = int(np.rint(np.clip(a[2], 0.0, 1.0)))

        did_cast = False
        cast_started = False
        can_turn_now = (not self.turn_requires_cast) or (self.turn_steps_left > 0)

        if self.in_cast:
            c = self._cast_step()
            self.v = 0.0
            self.omega = 0.0
            did_cast = True
        elif cast_cmd == 1:
            self.in_cast = True
            self.cast_phase = 0
            self._scan_c[:] = 0.0
            cast_started = True
            c = self._cast_step()
            self.v = 0.0
            self.omega = 0.0
            did_cast = True
        else:
            if not can_turn_now:
                omega_cmd = 0.0

            dv = np.clip(v_cmd - self.v, -self.accel_max * self.dt, self.accel_max * self.dt)
            domega = np.clip(omega_cmd - self.omega, -self.omega_accel_max * self.dt, self.omega_accel_max * self.dt)

            self.v = float(np.clip(self.v + dv, self.v_min, self.v_max))
            self.omega = float(np.clip(self.omega + domega, -self.omega_max, self.omega_max))

            self.th = self._norm_angle(self.th + self.omega * self.dt)
            self.x += self.v * np.cos(self.th) * self.dt
            self.y += self.v * np.sin(self.th) * self.dt

            c = self._sense(0.0)

            if self.turn_requires_cast and self.turn_steps_left > 0:
                self.turn_steps_left -= 1

        delta_c = float(c - prev_c)
        obs_mode = 1.0 if did_cast else 0.0
        self._set_obs(c, mode=obs_mode)

        oob = (abs(self.x) > self.L) or (abs(self.y) > self.L)
        terminated = bool(oob)
        truncated = bool(self._step >= self.max_steps)

        dx, dy = self.x - self.src_x, self.y - self.src_y
        d = np.hypot(dx, dy)
        in_goal = bool(d < self.r_goal)
        if in_goal:
            self.goal_hold_count += 1
        else:
            self.goal_hold_count = 0
        goal_exited = bool((not in_goal) and prev_in_goal)
        success_hold = bool(self.goal_hold_count >= self.goal_hold_steps)
        self.prev_in_goal = in_goal

        if self.reward_mode == "bio":
            # Bio reward: same structure as mechanical, but concentration shaping.
            r = float(self.bio_reward_scale * c)
            if did_cast:
                r -= self.cast_penalty
            else:
                r -= self.control_penalty * (abs(self.v) / (self.v_max + 1e-6))
                r -= self.turn_penalty * (abs(self.omega) / (self.omega_max + 1e-6))
            if in_goal:
                r += self.b_hold
        else:
            # Mechanical reward: distance shaping + action costs.
            r = np.exp(-d / self.sigma_r)
            if in_goal:
                r += self.b_hold
            if did_cast:
                r -= self.cast_penalty
            else:
                r -= self.control_penalty * (abs(self.v) / (self.v_max + 1e-6))
                r -= self.turn_penalty * (abs(self.omega) / (self.omega_max + 1e-6))
        if oob:
            r -= self.b_oob
        if self.terminate_on_hold and success_hold:
            terminated = True

        info = {
            "d": d,
            "c": float(c),
            "in_goal": int(in_goal),
            "goal_hold_count": int(self.goal_hold_count),
            "goal_exited": int(goal_exited),
            "success_hold": int(success_hold),
            "delta_c": delta_c,
            "src_x": self.src_x,
            "src_y": self.src_y,
            "v": self.v,
            "omega": self.omega,
            "did_cast": int(did_cast),
            "cast_start": int(cast_started),
            "cast_cmd": cast_cmd,
            "in_cast": int(self.in_cast),
            "can_turn": int(can_turn_now),
            "turn_steps_left": int(self.turn_steps_left),
        }
        self._trail.append((self.x, self.y))
        return self._get_obs(), float(r), terminated, truncated, info

    def _world_to_px(self, x, y):
        size = self._img_size - 1
        px = int(np.clip((float(x) + self.L) / (2.0 * self.L) * size, 0, size))
        py = int(np.clip((self.L - float(y)) / (2.0 * self.L) * size, 0, size))
        return px, py

    def _build_heatmap(self):
        size = self._img_size
        xs = np.linspace(-self.L, self.L, size, dtype=np.float32)
        ys = np.linspace(self.L, -self.L, size, dtype=np.float32)
        X, Y = np.meshgrid(xs, ys)
        c = self._conc(X, Y).astype(np.float32)
        try:
            from matplotlib import cm
            rgb = cm.inferno(c)[..., :3].astype(np.float32)
            return np.clip(rgb * 255.0, 0.0, 255.0).astype(np.uint8)
        except Exception:
            r = np.clip(4.0 * c - 1.5, 0.0, 1.0)
            g = np.clip(4.0 * c - 2.5, 0.0, 1.0)
            b = np.clip(4.0 * c - 3.5, 0.0, 1.0)
            return (np.stack([r, g, b], axis=-1) * 255.0).astype(np.uint8)

    def render(self):
        if self.render_mode not in (None, "rgb_array"):
            return None
        try:
            from PIL import Image, ImageDraw
        except Exception:
            if self._heatmap_img is None:
                self._heatmap_img = self._build_heatmap()
            return self._heatmap_img.copy()

        W = H = self._img_size
        if self._heatmap_img is None or self._heatmap_img.shape[:2] != (H, W):
            self._heatmap_img = self._build_heatmap()

        img = Image.fromarray(self._heatmap_img.copy(), mode="RGB")
        draw = ImageDraw.Draw(img)

        draw.rectangle((0, 0, W - 1, H - 1), outline=(255, 255, 255), width=1)

        src_px, src_py = self._world_to_px(self.src_x, self.src_y)
        rg = max(1, int(round(self.r_goal / (2.0 * self.L) * (W - 1))))
        draw.ellipse((src_px - rg, src_py - rg, src_px + rg, src_py + rg), outline=(190, 190, 190), width=2)
        draw.ellipse((src_px - 4, src_py - 4, src_px + 4, src_py + 4), fill=(0, 0, 0))

        if len(self._trail) > 1:
            pts = [self._world_to_px(tx, ty) for tx, ty in self._trail]
            draw.line(pts, fill=(80, 220, 255), width=2)

        ax, ay = self._world_to_px(self.x, self.y)
        size = 10
        p0 = (ax + size * np.cos(self.th), ay - size * np.sin(self.th))
        p1 = (ax + size * np.cos(self.th + 2.5), ay - size * np.sin(self.th + 2.5))
        p2 = (ax + size * np.cos(self.th - 2.5), ay - size * np.sin(self.th - 2.5))
        tri = [tuple(map(int, p0)), tuple(map(int, p1)), tuple(map(int, p2))]
        draw.polygon(tri, fill=(50, 100, 220))

        if self._sense_pt is not None:
            sx, sy = self._world_to_px(self._sense_pt[0], self._sense_pt[1])
            draw.line((ax, ay, sx, sy), fill=(220, 60, 60), width=2)
            draw.ellipse((sx - 3, sy - 3, sx + 3, sy + 3), fill=(220, 60, 60))

        return np.array(img, dtype=np.uint8)

    def close(self):
        self._heatmap_img = None
        self._cbar_img = None


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Bernoulli
from torch.distributions import Normal


# Connectome2: ORN:PN:LN:KC:MBON = 24:7:4:54:1, base unit 90.
CONNECTOME2_BASE = 90
CONNECTOME2_RATIOS = (24, 7, 4, 54, 1)  # ORN, PN, LN, KC, MBON


def _population_sizes_v2(total_size):
    """Split hidden size into ORN/PN/LN/KC/MBON using ratio 24:7:4:54:1."""
    if total_size % CONNECTOME2_BASE != 0:
        raise ValueError(
            f"Connectome2 hidden_size must be a multiple of {CONNECTOME2_BASE}; got {total_size}."
        )
    k = total_size // CONNECTOME2_BASE
    if k < 2:
        raise ValueError(
            f"Connectome2 requires hidden_size >= {2 * CONNECTOME2_BASE} (MBON >= 2); got {total_size}."
        )

    n_orn = CONNECTOME2_RATIOS[0] * k
    n_pn = CONNECTOME2_RATIOS[1] * k
    n_ln = CONNECTOME2_RATIOS[2] * k
    n_kc = CONNECTOME2_RATIOS[3] * k
    n_mbon = CONNECTOME2_RATIOS[4] * k
    return n_orn, n_pn, n_ln, n_kc, n_mbon


class _Connectome2Backbone(nn.Module):
    """
    Connectome2 recurrent backbone with five interacting populations:
    ORN, PN, LN, KC, MBON.
    Hidden state is packed as a single tensor so it fits RSAC actor API.
    """
    def __init__(self, obs_dim, hidden_size, inner_steps=4):
        super().__init__()
        self.hidden_size = int(hidden_size)
        self.inner_steps = int(max(1, inner_steps))

        n_orn, n_pn, n_ln, n_kc, n_mbon = _population_sizes_v2(self.hidden_size)
        self.n_orn = n_orn
        self.n_pn = n_pn
        self.n_ln = n_ln
        self.n_kc = n_kc
        self.n_mbon = n_mbon

        self.in_orn = nn.Linear(obs_dim, n_orn)

        # ORN <- ORN, LN, input
        self.W_oto = nn.Linear(n_orn, n_orn, bias=False)
        self.W_lto = nn.Linear(n_ln, n_orn, bias=False)

        # PN <- ORN, LN, PN
        self.W_otp = nn.Linear(n_orn, n_pn, bias=False)
        self.W_ltp = nn.Linear(n_ln, n_pn, bias=False)
        self.W_ptp = nn.Linear(n_pn, n_pn, bias=False)

        # LN <- ORN, PN, LN
        self.W_otl = nn.Linear(n_orn, n_ln, bias=False)
        self.W_ptl = nn.Linear(n_pn, n_ln, bias=False)
        self.W_ltl = nn.Linear(n_ln, n_ln, bias=False)

        # KC <- KC, MBON, PN
        self.W_ktk = nn.Linear(n_kc, n_kc, bias=False)
        self.W_mtk = nn.Linear(n_mbon, n_kc, bias=False)
        self.W_ptk = nn.Linear(n_pn, n_kc, bias=False)

        # MBON <- KC
        self.W_ktm = nn.Linear(n_kc, n_mbon, bias=False)

        self.b_orn = nn.Parameter(torch.zeros(n_orn))
        self.b_pn = nn.Parameter(torch.zeros(n_pn))
        self.b_ln = nn.Parameter(torch.zeros(n_ln))
        self.b_kc = nn.Parameter(torch.zeros(n_kc))
        self.b_mbon = nn.Parameter(torch.zeros(n_mbon))

        self.readout = nn.Linear(n_mbon, self.hidden_size)

    def _split_state(self, h_flat):
        i0 = self.n_orn
        i1 = i0 + self.n_pn
        i2 = i1 + self.n_ln
        i3 = i2 + self.n_kc
        h_orn = h_flat[:, :i0]
        h_pn = h_flat[:, i0:i1]
        h_ln = h_flat[:, i1:i2]
        h_kc = h_flat[:, i2:i3]
        h_mbon = h_flat[:, i3:]
        return h_orn, h_pn, h_ln, h_kc, h_mbon

    def _pack_state(self, h_orn, h_pn, h_ln, h_kc, h_mbon):
        return torch.cat([h_orn, h_pn, h_ln, h_kc, h_mbon], dim=-1)

    def _init_state(self, batch_size, device, dtype):
        return torch.zeros(batch_size, self.hidden_size, device=device, dtype=dtype)

    def _parse_hidden(self, h, batch_size, device, dtype):
        if h is None:
            return self._init_state(batch_size, device, dtype)
        if h.dim() == 3:
            if h.size(0) != 1:
                raise ValueError("Connectome2 hidden state expects first dim 1.")
            h_flat = h[0]
        elif h.dim() == 2:
            h_flat = h
        else:
            raise ValueError("Connectome2 hidden state must be rank-2 or rank-3 tensor.")
        if h_flat.size(0) != batch_size or h_flat.size(1) != self.hidden_size:
            raise ValueError(
                f"Connectome2 hidden state shape mismatch: got {tuple(h_flat.shape)}, "
                f"expected ({batch_size}, {self.hidden_size})."
            )
        return h_flat.to(device=device, dtype=dtype)

    def step(self, h_orn, h_pn, h_ln, h_kc, h_mbon, x_t):
        orn_next = torch.tanh(self.W_oto(h_orn) + self.W_lto(h_ln) + x_t + self.b_orn)
        pn_next = torch.tanh(self.W_otp(orn_next) + self.W_ltp(h_ln) + self.W_ptp(h_pn) + self.b_pn)
        ln_next = torch.tanh(self.W_otl(orn_next) + self.W_ptl(pn_next) + self.W_ltl(h_ln) + self.b_ln)
        kc_next = torch.tanh(self.W_ktk(h_kc) + self.W_mtk(h_mbon) + self.W_ptk(pn_next) + self.b_kc)
        mbon_next = torch.tanh(self.W_ktm(kc_next) + self.b_mbon)
        return orn_next, pn_next, ln_next, kc_next, mbon_next

    def forward(self, obs, h=None):
        if obs.dim() == 2:
            obs = obs.unsqueeze(1)
        bsz, seq_len, _ = obs.shape
        device = obs.device
        dtype = obs.dtype

        h_flat = self._parse_hidden(h, bsz, device, dtype)
        h_orn, h_pn, h_ln, h_kc, h_mbon = self._split_state(h_flat)

        outputs = []
        for t in range(seq_len):
            x_t = self.in_orn(obs[:, t, :])
            for _ in range(self.inner_steps):
                h_orn, h_pn, h_ln, h_kc, h_mbon = self.step(h_orn, h_pn, h_ln, h_kc, h_mbon, x_t)
            outputs.append(self.readout(h_mbon).unsqueeze(1))

        y = torch.cat(outputs, dim=1)
        h2 = self._pack_state(h_orn, h_pn, h_ln, h_kc, h_mbon).unsqueeze(0)
        return y, h2


class RecurrentGaussianActor(nn.Module):
    def __init__(self, obs_dim, act_dim, action_low, action_high, hidden=147, log_std_min=-5.0, log_std_max=2.0):
        super().__init__()
        self.rnn = _build_gru(obs_dim, hidden)
        self.mu = nn.Linear(hidden, act_dim)
        self.log_std = nn.Linear(hidden, act_dim)
        self.log_std_min = float(log_std_min)
        self.log_std_max = float(log_std_max)

        low = torch.as_tensor(action_low, dtype=torch.float32)
        high = torch.as_tensor(action_high, dtype=torch.float32)
        self.register_buffer("action_scale", (high - low) * 0.5)
        self.register_buffer("action_bias", (high + low) * 0.5)

    def forward(self, obs, h=None):
        if obs.dim() == 2:
            obs = obs.unsqueeze(1)
        y, h2 = self.rnn(obs, h)
        mu = self.mu(y)
        log_std = torch.clamp(self.log_std(y), self.log_std_min, self.log_std_max)
        return mu, log_std, h2

    def sample(self, obs, h=None):
        mu, log_std, h2 = self.forward(obs, h)
        std = log_std.exp()
        normal = Normal(mu, std)
        x = normal.rsample()
        y = torch.tanh(x)
        action = y * self.action_scale + self.action_bias

        # tanh-squashed Gaussian log-prob with change-of-variables correction
        log_prob = normal.log_prob(x) - torch.log(self.action_scale * (1.0 - y.pow(2)) + 1e-6)
        log_prob = log_prob.sum(dim=-1)
        return action, log_prob, h2, mu

    def deterministic(self, obs, h=None):
        mu, _, h2 = self.forward(obs, h)
        y = torch.tanh(mu)
        action = y * self.action_scale + self.action_bias
        return action, h2


class RecurrentQCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=147):
        super().__init__()
        self.rnn = _build_gru(obs_dim, hidden)
        self.fc1 = nn.Linear(hidden + act_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.q = nn.Linear(hidden, 1)

    def forward(self, obs, act, h=None):
        if obs.dim() == 2:
            obs = obs.unsqueeze(1)
        if act.dim() == 2:
            act = act.unsqueeze(1)
        y, h2 = self.rnn(obs, h)
        z = torch.cat([y, act], dim=-1)
        z = F.relu(self.fc1(z))
        z = F.relu(self.fc2(z))
        q = self.q(z).squeeze(-1)
        return q, h2

class Connectome2HybridActor(nn.Module):
    """
    Hybrid policy with Connectome2 recurrent backbone for [v, omega, cast].
    The policy head API is identical to RecurrentHybridActor.
    """
    def __init__(
        self,
        obs_dim,
        cont_act_dim,
        hidden=180,
        log_std_min=-5.0,
        log_std_max=2.0,
        cast_temperature=0.5,
        connectome_steps=4,
    ):
        super().__init__()
        self.backbone = _Connectome2Backbone(
            obs_dim=obs_dim,
            hidden_size=hidden,
            inner_steps=connectome_steps,
        )
        self.mu = nn.Linear(hidden, cont_act_dim)
        self.log_std = nn.Linear(hidden, cont_act_dim)
        self.cast_logit = nn.Linear(hidden, 1)
        self.log_std_min = float(log_std_min)
        self.log_std_max = float(log_std_max)
        self.cast_temperature = float(max(cast_temperature, 1e-3))

    def forward(self, obs, h=None):
        if obs.dim() == 2:
            obs = obs.unsqueeze(1)
        y, h2 = self.backbone(obs, h)
        mu = self.mu(y)
        log_std = torch.clamp(self.log_std(y), self.log_std_min, self.log_std_max)
        cast_logit = self.cast_logit(y)
        return mu, log_std, cast_logit, h2

    def sample(self, obs, action_low, action_high, h=None):
        mu, log_std, cast_logit, h2 = self.forward(obs, h)
        std = log_std.exp()
        normal = Normal(mu, std)
        x = normal.rsample()
        y = torch.tanh(x)

        cont_low = action_low[:2]
        cont_high = action_high[:2]
        cont_scale = (cont_high - cont_low) * 0.5
        cont_bias = (cont_high + cont_low) * 0.5
        cont_action = y * cont_scale + cont_bias

        cont_log_prob = normal.log_prob(x) - torch.log(cont_scale * (1.0 - y.pow(2)) + 1e-6)
        cont_log_prob = cont_log_prob.sum(dim=-1, keepdim=True)

        u = torch.rand_like(cast_logit)
        u = torch.clamp(u, 1e-6, 1.0 - 1e-6)
        logistic_noise = torch.log(u) - torch.log1p(-u)
        cast_soft = torch.sigmoid((cast_logit + logistic_noise) / self.cast_temperature)
        cast_hard = (cast_soft > 0.5).float()
        cast_action = cast_hard + cast_soft - cast_soft.detach()

        bern = Bernoulli(logits=cast_logit)
        disc_log_prob = bern.log_prob(cast_hard)

        action = torch.cat([cont_action, cast_action], dim=-1)
        log_prob = (cont_log_prob + disc_log_prob).squeeze(-1)
        cast_prob = torch.sigmoid(cast_logit)
        return action, log_prob, h2, mu, cast_prob

    def deterministic(self, obs, action_low, action_high, h=None):
        mu, _, cast_logit, h2 = self.forward(obs, h)
        y = torch.tanh(mu)

        cont_low = action_low[:2]
        cont_high = action_high[:2]
        cont_scale = (cont_high - cont_low) * 0.5
        cont_bias = (cont_high + cont_low) * 0.5
        cont_action = y * cont_scale + cont_bias

        cast_action = (cast_logit > 0.0).float()
        action = torch.cat([cont_action, cast_action], dim=-1)
        return action, h2


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim



class RSACAgent:
    def __init__(
        self,
        obs_dim,
        act_dim,
        action_low,
        action_high,
        device,
        rnn_hidden=147,
        lr_actor=3e-4,
        lr_critic=3e-4,
        lr_alpha=3e-4,
        gamma=0.99,
        tau=0.005,
        actor_backbone="gru",
        connectome_steps=4,
        connectome_hidden=180,
    ):
        self.device = device
        self.gamma = float(gamma)
        self.tau = float(tau)
        self.act_dim = int(act_dim)
        self.actor_backbone = str(actor_backbone).lower()
        if self.act_dim != 3:
            raise ValueError("RSACAgent expects action dim exactly 3: [v, omega, cast].")

        self.action_low = torch.as_tensor(action_low, dtype=torch.float32, device=device)
        self.action_high = torch.as_tensor(action_high, dtype=torch.float32, device=device)
        if self.action_low.numel() != 3 or self.action_high.numel() != 3:
            raise ValueError("RSACAgent expects action bounds with 3 elements: [v, omega, cast].")

        if self.actor_backbone == "connectome":
            self.actor = ConnectomeHybridActor(
                obs_dim,
                cont_act_dim=2,
                hidden=connectome_hidden,
                connectome_steps=connectome_steps,
            ).to(device)
        elif self.actor_backbone == "connectome2":
            self.actor = Connectome2HybridActor(
                obs_dim,
                cont_act_dim=2,
                hidden=connectome_hidden,
                connectome_steps=connectome_steps,
            ).to(device)
        elif self.actor_backbone == "gru":
            self.actor = RecurrentHybridActor(
                obs_dim,
                cont_act_dim=2,
                hidden=rnn_hidden,
            ).to(device)
        else:
            raise ValueError(
                f"Unsupported actor_backbone: {self.actor_backbone}. "
                "Use 'gru', 'connectome', or 'connectome2'."
            )

        self.q1 = RecurrentQCritic(obs_dim, act_dim, hidden=rnn_hidden).to(device)
        self.q2 = RecurrentQCritic(obs_dim, act_dim, hidden=rnn_hidden).to(device)
        self.tq1 = RecurrentQCritic(obs_dim, act_dim, hidden=rnn_hidden).to(device)
        self.tq2 = RecurrentQCritic(obs_dim, act_dim, hidden=rnn_hidden).to(device)
        self.tq1.load_state_dict(self.q1.state_dict())
        self.tq2.load_state_dict(self.q2.state_dict())

        self.actor_opt = optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.critic_opt = optim.Adam(list(self.q1.parameters()) + list(self.q2.parameters()), lr=lr_critic)

        self.log_alpha = torch.zeros(1, device=device, requires_grad=True)
        self.alpha_opt = optim.Adam([self.log_alpha], lr=lr_alpha)
        # Hybrid entropy target:
        # - continuous [v, omega]: SAC default ~= -dim_cont
        # - binary cast action: -0.5 * log(2) (encourages some stochasticity without dominating)
        self.target_entropy = -(2.0 + 0.5 * float(np.log(2.0)))

        self.loss_fn = nn.SmoothL1Loss()

    @property
    def alpha(self):
        return self.log_alpha.exp()

    def get_action(self, obs, h=None, epsilon=1.0):
        del epsilon
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            action, _, h2, _, _ = self.actor.sample(obs_t, self.action_low, self.action_high, h)
        a = action[:, -1, :].squeeze(0).detach().cpu().numpy().astype(np.float32)
        return a, h2

    def get_action_deterministic(self, obs, h=None):
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            action, h2 = self.actor.deterministic(obs_t, self.action_low, self.action_high, h)
        a = action[:, -1, :].squeeze(0).detach().cpu().numpy().astype(np.float32)
        return a, h2

    def update(self, batch):
        obs_seq, act_seq, rew_seq, terminal_seq = batch

        obs_seq = torch.as_tensor(obs_seq, dtype=torch.float32, device=self.device)    # (B, T+1, D)
        act_seq = torch.as_tensor(act_seq, dtype=torch.float32, device=self.device)    # (B, T, A)
        rew_seq = torch.as_tensor(rew_seq, dtype=torch.float32, device=self.device)    # (B, T)
        terminal_seq = torch.as_tensor(terminal_seq, dtype=torch.float32, device=self.device)  # (B, T)

        obs_t = obs_seq[:, :-1, :]
        next_obs_t = obs_seq[:, 1:, :]

        with torch.no_grad():
            next_a, next_logp, _, _, _ = self.actor.sample(next_obs_t, self.action_low, self.action_high, None)
            next_q1, _ = self.tq1(next_obs_t, next_a, None)
            next_q2, _ = self.tq2(next_obs_t, next_a, None)
            next_q = torch.min(next_q1, next_q2) - self.alpha.detach() * next_logp
            y = rew_seq + self.gamma * (1.0 - terminal_seq) * next_q

        q1, _ = self.q1(obs_t, act_seq, None)
        q2, _ = self.q2(obs_t, act_seq, None)
        critic_loss = self.loss_fn(q1, y) + self.loss_fn(q2, y)

        self.critic_opt.zero_grad()
        critic_loss.backward()
        self.critic_opt.step()

        pi_a, logp, _, _, _ = self.actor.sample(obs_t, self.action_low, self.action_high, None)
        q1_pi, _ = self.q1(obs_t, pi_a, None)
        q2_pi, _ = self.q2(obs_t, pi_a, None)
        q_pi = torch.min(q1_pi, q2_pi)
        actor_loss = (self.alpha.detach() * logp - q_pi).mean()

        self.actor_opt.zero_grad()
        actor_loss.backward()
        self.actor_opt.step()

        alpha_loss = -(self.log_alpha * (logp.detach() + self.target_entropy)).mean()
        self.alpha_opt.zero_grad()
        alpha_loss.backward()
        self.alpha_opt.step()

        self.soft_update()

        td_err = (y - q1).abs().mean().item()
        return {
            "critic_loss": float(critic_loss.item()),
            "actor_loss": float(actor_loss.item()),
            "alpha_loss": float(alpha_loss.item()),
            "alpha": float(self.alpha.item()),
            "td_abs": float(td_err),
        }

    def soft_update(self):
        with torch.no_grad():
            for p, tp in zip(self.q1.parameters(), self.tq1.parameters()):
                tp.data.mul_(1.0 - self.tau).add_(self.tau * p.data)
            for p, tp in zip(self.q2.parameters(), self.tq2.parameters()):
                tp.data.mul_(1.0 - self.tau).add_(self.tau * p.data)

    def sync_target(self):
        # compatibility with existing training loop; SAC uses soft updates each step.
        return None

    def save(self, path):
        ckpt = {
            "actor": self.actor.state_dict(),
            "actor_backbone": self.actor_backbone,
            "q1": self.q1.state_dict(),
            "q2": self.q2.state_dict(),
            "tq1": self.tq1.state_dict(),
            "tq2": self.tq2.state_dict(),
            "log_alpha": self.log_alpha.detach().cpu(),
        }
        torch.save(ckpt, path)

    def load(self, path):
        ckpt = torch.load(path, map_location=self.device)
        ckpt_backbone = str(ckpt.get("actor_backbone", "gru")).lower()
        if str(ckpt_backbone).lower() != self.actor_backbone:
            raise ValueError(
                f"Checkpoint actor_backbone={ckpt_backbone} but agent actor_backbone={self.actor_backbone}."
            )
        self.actor.load_state_dict(ckpt["actor"])
        self.q1.load_state_dict(ckpt["q1"])
        self.q2.load_state_dict(ckpt["q2"])
        self.tq1.load_state_dict(ckpt.get("tq1", ckpt["q1"]))
        self.tq2.load_state_dict(ckpt.get("tq2", ckpt["q2"]))
        if "log_alpha" in ckpt:
            self.log_alpha.data.copy_(ckpt["log_alpha"].to(self.device))


In [ ]:
import os
import json
import time
import numpy as np
import torch


def train_rsac(args):
    set_global_seed(args["seed"])
    run_name = args["run_name"]
    out_dir = args["out_dir"]
    run_dir = os.path.abspath(os.path.join(out_dir, run_name))
    ckpt_dir = os.path.join(run_dir, "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)

    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(args, f, indent=2)

    device = torch.device("cuda" if torch.cuda.is_available() and not args["force_cpu"] else "cpu")
    print("device:", device)

    env = OdorHoldEnvV4(
        reward_mode=args["reward_mode"],
        bio_reward_scale=args["bio_reward_scale"],
        cast_penalty=args["cast_penalty"],
        turn_penalty=args["turn_penalty"],
        b_hold=args["b_hold"],
        goal_hold_steps=args["goal_hold_steps"],
        terminate_on_hold=args["terminate_on_hold"],
        sigma_c=args["sigma_c"],
        src_x=args["src_x"],
        src_y=args["src_y"],
        wind_x=args["wind_x"],
        seed=args["seed"],
    )
    env.action_space.seed(args["seed"])
    if hasattr(env.observation_space, "seed"):
        env.observation_space.seed(args["seed"])

    agent = RSACAgent(
        obs_dim=env.observation_space.shape[0],
        act_dim=env.action_space.shape[0],
        action_low=env.action_space.low,
        action_high=env.action_space.high,
        device=device,
        rnn_hidden=args["rnn_hidden"],
        lr_actor=args["lr_actor"],
        lr_critic=args["lr_critic"],
        lr_alpha=args["lr_alpha"],
        gamma=args["gamma"],
        tau=args["tau"],
        actor_backbone=args["rsac_actor_backbone"],
        connectome_steps=args["connectome_steps"],
        connectome_hidden=args["connectome_hidden"],
    )

    buffer = EpisodeReplayBuffer(args["buffer_size"])

    best_return = -np.inf
    returns = []

    for ep in range(1, args["total_episodes"] + 1):
        obs, _ = env.reset(seed=args["seed"] + ep)
        h = None
        done = False
        ep_ret = 0.0
        traj = []
        log_stats = None

        while not done:
            action, h_next = agent.get_action(obs, h, epsilon=1.0)
            next_obs, reward, term, trunc, _ = env.step(action)
            done = bool(term or trunc)
            terminal = float(term)

            traj.append((obs, action, float(reward), next_obs, terminal))
            obs = next_obs
            h = h_next
            ep_ret += float(reward)

            if len(buffer) > args["learning_starts"] and len(buffer) > args["batch_size"] * args["seq_len"]:
                batch = buffer.sample_continuous(args["batch_size"], args["seq_len"])
                log_stats = agent.update(batch)

        buffer.add_episode(traj)
        returns.append(ep_ret)

        if ep_ret > best_return:
            best_return = ep_ret
            agent.save(os.path.join(ckpt_dir, "best.pt"))

        if ep % args["log_every"] == 0:
            avg_ret = float(np.mean(returns[-args["log_every"]:]))
            if isinstance(log_stats, dict):
                print(
                    f"Ep {ep} | Avg Ret: {avg_ret:.2f} | alpha: {log_stats['alpha']:.4f} | td|: {log_stats['td_abs']:.4f}"
                )
            else:
                print(f"Ep {ep} | Avg Ret: {avg_ret:.2f}")

    agent.save(os.path.join(ckpt_dir, "final.pt"))
    env.close()

    return {
        "run_dir": run_dir,
        "best_return": float(best_return),
    }


def evaluate_rsac(run_dir, episodes=50, seed=42, force_cpu=False):
    with open(os.path.join(run_dir, "config.json"), "r") as f:
        conf = json.load(f)

    device = torch.device("cuda" if torch.cuda.is_available() and not force_cpu else "cpu")
    print("eval device:", device)

    env = OdorHoldEnvV4(
        reward_mode=conf["reward_mode"],
        bio_reward_scale=conf["bio_reward_scale"],
        cast_penalty=conf["cast_penalty"],
        turn_penalty=conf["turn_penalty"],
        b_hold=conf["b_hold"],
        goal_hold_steps=conf["goal_hold_steps"],
        terminate_on_hold=conf["terminate_on_hold"],
        sigma_c=conf["sigma_c"],
        src_x=conf["src_x"],
        src_y=conf["src_y"],
        wind_x=conf["wind_x"],
        seed=seed,
    )

    agent = RSACAgent(
        obs_dim=env.observation_space.shape[0],
        act_dim=env.action_space.shape[0],
        action_low=env.action_space.low,
        action_high=env.action_space.high,
        device=device,
        rnn_hidden=conf["rnn_hidden"],
        lr_actor=conf["lr_actor"],
        lr_critic=conf["lr_critic"],
        lr_alpha=conf["lr_alpha"],
        gamma=conf["gamma"],
        tau=conf["tau"],
        actor_backbone=conf["rsac_actor_backbone"],
        connectome_steps=conf["connectome_steps"],
        connectome_hidden=conf["connectome_hidden"],
    )

    ckpt_path = os.path.join(run_dir, "checkpoints", "best.pt")
    if not os.path.exists(ckpt_path):
        ckpt_path = os.path.join(run_dir, "checkpoints", "final.pt")
    print("loading:", ckpt_path)
    agent.load(ckpt_path)

    returns = []
    success_entry = 0
    success_hold = 0

    for i in range(episodes):
        obs, _ = env.reset(seed=20000 + i)
        h = None
        done = False
        ep_ret = 0.0
        entered = False
        held = False

        while not done:
            action, h = agent.get_action_deterministic(obs, h)
            obs, r, term, trunc, info = env.step(action)
            done = bool(term or trunc)
            ep_ret += float(r)
            if info.get("in_goal", 0):
                entered = True
            if info.get("success_hold", 0):
                held = True

        returns.append(ep_ret)
        success_entry += int(entered)
        success_hold += int(held)

    env.close()

    metrics = {
        "episodes": int(episodes),
        "avg_return": float(np.mean(returns) if returns else 0.0),
        "entry_success_rate": float(success_entry / episodes if episodes > 0 else 0.0),
        "hold_success_rate": float(success_hold / episodes if episodes > 0 else 0.0),
    }

    print("metrics:")
    for k, v in metrics.items():
        if "rate" in k:
            print(f"  {k}: {v * 100:.1f}%")
        else:
            print(f"  {k}: {v}")

    with open(os.path.join(run_dir, "eval_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    return metrics


In [ ]:
# Connectome2 all-in-one run config
import time

args = {
    "out_dir": "runs",
    "run_name": f"connectome2_allinone_{time.strftime('%Y%m%d_%H%M%S')}",
    "seed": 42,
    "force_cpu": False,

    # env/reward
    "reward_mode": "bio",
    "bio_reward_scale": 0.5,
    "cast_penalty": 0.025,
    "turn_penalty": 0.01,
    "b_hold": 0.5,
    "goal_hold_steps": 20,
    "terminate_on_hold": True,
    "sigma_c": 1.0,
    "src_x": 0.0,
    "src_y": 0.0,
    "wind_x": 0.0,

    # rsac
    "total_episodes": 500,
    "batch_size": 128,
    "seq_len": 6,
    "buffer_size": 150000,
    "learning_starts": 5000,
    "log_every": 20,
    "rnn_hidden": 147,
    "lr_actor": 3e-4,
    "lr_critic": 3e-4,
    "lr_alpha": 3e-4,
    "gamma": 0.99,
    "tau": 0.005,

    # backbone fixed for this notebook
    "rsac_actor_backbone": "connectome2",
    "connectome_hidden": 180,
    "connectome_steps": 4,
}

print("run_name:", args["run_name"])


In [ ]:
train_result = train_rsac(args)
run_dir = train_result["run_dir"]
print("run_dir:", run_dir)


In [ ]:
metrics = evaluate_rsac(
    run_dir=run_dir,
    episodes=50,
    seed=args["seed"],
    force_cpu=args["force_cpu"],
)
metrics


In [ ]:
from pathlib import Path

p = Path(run_dir)
print("saved artifacts:")
for x in sorted(p.glob("*")):
    print(" -", x)
